# 4D-STEM NBD Grain Segmentation and {111}c Planar-Defect Classification — Part 2

Loads the workspace state saved by **Part 1** and performs:
defect-grain selection → binarized defect map → virtual BF/DF/HAADF overlays → per-ROI defect-grain statistics (count & area ratios) → aggregated Gua20 vs Gua25 summary.

**Defect-grain criterion:** a grain is counted as defect-bearing only if its representative NBD pattern shows a {111}c diffraction peak together with a streak along the same direction. Diffraction patterns unambiguously assignable to δ-phase polytypes (long-range ordering of face-sharing octahedra) are excluded from the defect-grain count.

Set `defect_indices` below to the grain indices classified as defect-bearing for this ROI (determined from the grain-by-grain diffraction-pattern inspection).

In [ ]:
# ==============================================================================
# Load workspace state from Part 1
# ==============================================================================
import pickle
import os
import matplotlib.pyplot as plt
import numpy as np

dataset_name = "Gua20_Acq1"
state_path = f"./results/{dataset_name}_state.pkl"
if not os.path.exists(state_path):
    raise FileNotFoundError(
        f"State file not found. Run the Part 1 notebook for {dataset_name} first.")

with open(state_path, "rb") as f:
    state = pickle.load(f)
globals().update(state)
print(f"Workspace state loaded from: {state_path}")

In [ ]:
# ==================================================
# Defect-grain selection & overlay (inferno scale)
# ==================================================
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

# Grain indices classified as defect-bearing for THIS ROI.
# (From grain-by-grain NBD inspection: {111}c peak + co-directional streak;
#  delta-phase polytype patterns are excluded.)
defect_indices = [28, 92, 152]

if num_grains > 0 and len(defect_indices) > 0:
    H, W_grid = modified_data.shape[0], modified_data.shape[1]
    combined_defect_image = np.zeros((H, W_grid))
    for idx in defect_indices:
        if 0 <= idx < num_grains:
            combined_defect_image += all_grains[idx]["feature"]
        else:
            print(f"Warning: index {idx} out of range (0..{num_grains-1}).")
    if combined_defect_image.max() > 0:
        combined_defect_image = combined_defect_image / combined_defect_image.max()
    combined_defect_image = np.clip(combined_defect_image, 0, 1)

    fig = plt.figure(figsize=(10, 10))
    plt.imshow(combined_defect_image, cmap="inferno")
    plt.axis("off")
    plt.show()
    plt.close(fig)
else:
    print("No valid grains, or defect_indices is empty.")

In [ ]:
# ==================================================
# Binarized defect map (aligned with cleaned grain boundaries)
# ==================================================
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

if "L" in globals() and num_grains > 0 and len(defect_indices) > 0:
    # L holds labels 1..num_grains, so grain index idx maps to label idx+1.
    binary_defect_image = np.isin(L, [idx + 1 for idx in defect_indices]).astype(float)
    fig = plt.figure(figsize=(10, 10))
    plt.imshow(binary_defect_image, cmap="gray")
    plt.axis("off")
    plt.show()
    plt.close(fig)
else:
    print("Defect-map data unavailable. Run the previous cell first.")

In [ ]:
# ==================================================
# Virtual BF / DF / HAADF images overlaid with defect mask & grain boundaries
# ==================================================
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import numpy as np

alpha = 0.4              # red overlay transparency (0..1)
df_inner_radius   = 5.0  # virtual DF annulus, inner radius (pixels)
df_outer_radius   = 20.0 # virtual DF annulus, outer radius
haadf_inner_radius = 30.0
haadf_outer_radius = 60.0

if "binary_defect_image" in globals() and "raw_crop" in globals() and "circular_mask" in globals():
    # Recompute cleaned grain-boundary segments if needed.
    if "segments_clean" not in globals() and "L_clean" in globals():
        H_L, W_L = L_clean.shape
        seg = []
        for y in range(H_L - 1):
            for x in range(W_L):
                if L_clean[y, x] != L_clean[y + 1, x]:
                    seg.append([(x - 0.5, y + 0.5), (x + 0.5, y + 0.5)])
        for y in range(H_L):
            for x in range(W_L - 1):
                if L_clean[y, x] != L_clean[y, x + 1]:
                    seg.append([(x + 0.5, y - 0.5), (x + 0.5, y + 0.5)])
        globals()["segments_clean"] = seg

    H, W_grid = modified_data.shape[0], modified_data.shape[1]
    defect_mask = binary_defect_image == 1.0

    Q_Ny, Q_Nx = raw_crop.shape[2], raw_crop.shape[3]
    y_grid, x_grid = np.ogrid[:Q_Ny, :Q_Nx]
    dist_from_center = np.sqrt((y_grid - mask_center_y)**2 + (x_grid - mask_center_x)**2)

    def virtual_image(annulus_mask):
        v = np.sum(raw_crop * annulus_mask[None, None, :, :], axis=(2, 3))
        v = (v - v.min()) / (v.max() - v.min() + 1e-8)
        return np.stack([v]*3, axis=-1)

    def overlay(rgb):
        out = rgb.copy()
        out[defect_mask] = (1 - alpha) * rgb[defect_mask] + alpha * np.array([1.0, 0.0, 0.0])
        return np.clip(out, 0, 1)

    # Virtual BF (direct beam).
    bf_rgb = virtual_image(circular_mask)
    overlay_bf = overlay(bf_rgb)

    fig = plt.figure(figsize=(10, 10)); ax = fig.add_subplot(111)
    ax.imshow(overlay_bf)
    if "segments_clean" in globals():
        ax.add_collection(LineCollection(segments_clean, colors="white", linewidths=1.0))
    ax.axis("off")
    plt.savefig(save_prefix + f"_planar_defect_bf_overlay_alpha{alpha}.png",
                bbox_inches="tight", dpi=150)
    print("Saved virtual-BF overlay.")
    plt.show(); plt.close(fig)

    # Virtual HAADF (high-angle annulus).
    haadf_mask = (dist_from_center >= haadf_inner_radius) & (dist_from_center <= haadf_outer_radius)
    overlay_haadf = overlay(virtual_image(haadf_mask))
    fig = plt.figure(figsize=(10, 10)); ax = fig.add_subplot(111)
    ax.imshow(overlay_haadf)
    if "segments_clean" in globals():
        ax.add_collection(LineCollection(segments_clean, colors="white", linewidths=1.0))
    ax.axis("off")
    plt.savefig(save_prefix + f"_planar_defect_haadf_overlay_alpha{alpha}.png",
                bbox_inches="tight", dpi=150)
    print("Saved virtual-HAADF overlay.")
    plt.show(); plt.close(fig)

    # Virtual DF (low-angle annulus).
    df_mask = (dist_from_center >= df_inner_radius) & (dist_from_center <= df_outer_radius)
    overlay_df = overlay(virtual_image(df_mask))
    fig = plt.figure(figsize=(10, 10)); ax = fig.add_subplot(111)
    ax.imshow(overlay_df)
    if "segments_clean" in globals():
        ax.add_collection(LineCollection(segments_clean, colors="white", linewidths=1.0))
    ax.axis("off")
    plt.savefig(save_prefix + f"_planar_defect_df_overlay_alpha{alpha}.png",
                bbox_inches="tight", dpi=150)
    print("Saved virtual-DF overlay.")
    plt.show(); plt.close(fig)
else:
    print("Required variables (raw_crop, circular_mask, binary_defect_image) not found.")

In [ ]:
# ==============================================================================
# Per-ROI defect-grain statistics (count & area ratios) and save to JSON
# ==============================================================================
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import os
import json

if "all_grains" in globals() and "defect_indices" in globals() and len(all_grains) > 0:
    num_total_grains = len(all_grains)
    valid_defect_indices = [idx for idx in defect_indices if 0 <= idx < num_total_grains]
    num_defect_grains = len(valid_defect_indices)

    all_areas    = [g["area"] for g in all_grains]
    defect_areas = [all_grains[idx]["area"] for idx in valid_defect_indices]

    total_area_all    = sum(all_areas)
    total_area_defect = sum(defect_areas)

    count_defect_ratio = (num_defect_grains / num_total_grains) * 100 if num_total_grains else 0
    area_defect_ratio  = (total_area_defect / total_area_all) * 100 if total_area_all else 0

    print(f"=== Defect-grain statistics ({dataset_name}) ===")
    print(f"  total grains:  {num_total_grains}")
    print(f"  defect grains: {num_defect_grains}")
    print(f"  count ratio:   {count_defect_ratio:.2f}%")
    print(f"  area ratio:    {area_defect_ratio:.2f}%")

    # Persist per-ROI statistics.
    stats_file = "./results/statistics.json"
    stats_data = {}
    if os.path.exists(stats_file):
        try:
            with open(stats_file, "r", encoding="utf-8") as sf:
                stats_data = json.load(sf)
        except Exception:
            pass
    stats_data[dataset_name] = {
        "count_defect_ratio": float(count_defect_ratio),
        "area_defect_ratio": float(area_defect_ratio),
        "num_defect_grains": int(num_defect_grains),
        "num_total_grains": int(num_total_grains),
        "total_area_defect": int(total_area_defect),
        "total_area_all": int(total_area_all),
    }
    with open(stats_file, "w", encoding="utf-8") as sf:
        json.dump(stats_data, sf, indent=4, ensure_ascii=False)
    print(f"Saved per-ROI statistics to: {stats_file}")
else:
    print("Cannot compute statistics. Run the NMF / defect-selection cells first.")

In [ ]:
# ==============================================================================
# Aggregated Gua20 vs Gua25 summary (mean +/- s.d. over ROIs), count & area
# ==============================================================================
%matplotlib inline
import os, json
import numpy as np
import matplotlib.pyplot as plt

# ROI keys per composition (edit if your acquisition labels differ).
gua20_keys = [f"Gua20_Acq{i}" for i in range(1, 5)]
gua25_keys = ["Gua25_Acq1", "Gua25_Acq2", "Gua25_Acq3", "Gua25_Acq5"]

stats_file = "./results/statistics.json"
if os.path.exists(stats_file):
    with open(stats_file, "r", encoding="utf-8") as f:
        stats_data = json.load(f)

    def collect(keys, field):
        return [stats_data[k][field] for k in keys if k in stats_data]

    g20_counts = collect(gua20_keys, "count_defect_ratio")
    g25_counts = collect(gua25_keys, "count_defect_ratio")
    g20_areas  = collect(gua20_keys, "area_defect_ratio")
    g25_areas  = collect(gua25_keys, "area_defect_ratio")

    def msd(v):
        return (np.mean(v), np.std(v)) if len(v) else (0.0, 0.0)

    fig, (ax_c, ax_a) = plt.subplots(1, 2, figsize=(7, 3.5))
    labels = ["Gua20", "Gua25"]

    for ax, (v20, v25), ylab in [
        (ax_c, (g20_counts, g25_counts), "Defect Grain Count Ratio (%)"),
        (ax_a, (g20_areas,  g25_areas),  "Defect Grain Area Ratio (%)"),
    ]:
        means = [msd(v20)[0], msd(v25)[0]]
        stds  = [msd(v20)[1], msd(v25)[1]]
        ax.bar(labels, means, yerr=stds, align="center", alpha=0.8,
               color=["dodgerblue", "crimson"], edgecolor="black", capsize=10, width=0.4)
        ax.set_ylabel(ylab, fontsize=11)
        ax.grid(axis="y", linestyle="--", alpha=0.7)
        # Overlay individual ROI points.
        for x_val, v in zip([0, 1], [v20, v25]):
            for yv in v:
                ax.scatter(x_val, yv, color="black", edgecolor="white", zorder=5, s=50)
        ax.set_xlim(-0.5, 1.5)

    ax_a.yaxis.set_label_position("right"); ax_a.yaxis.tick_right()
    plt.tight_layout()
    out = "./results/summary_statistics_combined_side_by_side.png"
    plt.savefig(out, dpi=200, bbox_inches="tight")
    print(f"Saved summary figure to: {out}")
    for name, c, a in [("Gua20", g20_counts, g20_areas), ("Gua25", g25_counts, g25_areas)]:
        print(f"{name}: count {msd(c)[0]:.2f} +/- {msd(c)[1]:.2f} %  |  area {msd(a)[0]:.2f} +/- {msd(a)[1]:.2f} %")
    plt.show(); plt.close(fig)
else:
    print("statistics.json not found. Run the per-ROI statistics cell for each ROI first.")